
# Excel Lokomotif → Pandas → SQLite

Notebook ini dibuat khusus untuk format Excel **DAFTAR NOMOR EQUIPMENT LOKOMOTIF** seperti screenshot Anda.

File sumber berbentuk **report layout**, bukan tabel datar biasa. Dalam satu worksheet dapat ada beberapa blok lokomotif secara horizontal, misalnya:

```text
Blok 1: A:G
Blok 2: I:O
```

Setiap blok berisi metadata:

- NO.SERI LOKOMOTIF
- DIPO INDUK
- JENIS PERAWATAN
- PROGRAM BULAN
- MASUK
- KELUAR

dan detail komponen:

- NO.
- NAMA KOMPONEN
- KOMPONEN ASAL → KODE CETAK, NO.MANUF
- KOMPONEN PENGGANTI → KODE CETAK, NO.MANUF
- KET.

Target normalisasi:

```text
maintenance_events
        |
        | 1:N
        v
equipment_components
```

Artinya satu blok lokomotif menjadi satu maintenance event, lalu setiap equipment/component menjadi child rows.



## 1. Import library


In [1]:

# Uncomment jika belum terinstall:
# %pip install pandas openpyxl

import pandas as pd
import numpy as np
import sqlite3
import re

from pathlib import Path
from datetime import datetime, timezone

print("Pandas:", pd.__version__)
print("SQLite:", sqlite3.sqlite_version)


Pandas: 2.2.2
SQLite: 3.45.3



## 2. Configuration

Ubah `EXCEL_FILE` sesuai nama file asli Anda.

`BLOCK_WIDTH = 7` karena tabel komponen pada screenshot memiliki 7 kolom per blok.


In [ ]:

EXCEL_FILE = Path("Daftar Nomor Equipment Lokomotif 2026.xlsx")
DB_FILE = Path("kai.db")

# None = semua worksheet
SHEETS_TO_PROCESS = None

BLOCK_WIDTH = 7

print("Excel:", EXCEL_FILE)
print("DB:", DB_FILE)



## 3. Validasi file


In [ ]:

if not EXCEL_FILE.exists():
    print("File belum ditemukan:", EXCEL_FILE.resolve())
else:
    print("File ditemukan:", EXCEL_FILE.resolve())



## 4. Inspect worksheet


In [ ]:

if EXCEL_FILE.exists():
    xls = pd.ExcelFile(EXCEL_FILE, engine="openpyxl")

    print("Sheets:")
    for i, sheet in enumerate(xls.sheet_names, 1):
        print(f"{i}. {sheet}")

    if SHEETS_TO_PROCESS is None:
        selected_sheets = xls.sheet_names
    else:
        selected_sheets = SHEETS_TO_PROCESS



# PART A — Read Excel sebagai raw grid

Karena Excel ini bukan tabel sederhana, gunakan `header=None`.

Ini menjaga posisi cell agar parsing merged cells dan multiple horizontal blocks tetap memungkinkan.


In [ ]:

if EXCEL_FILE.exists():
    sample_sheet = selected_sheets[0]

    raw = pd.read_excel(
        EXCEL_FILE,
        sheet_name=sample_sheet,
        header=None,
        dtype=object,
        engine="openpyxl"
    )

    print("Sheet:", sample_sheet)
    print("Shape:", raw.shape)

    display(raw.head(50))



## 5. Helper cleaning cell


In [ ]:

def clean_cell(value):
    if pd.isna(value):
        return None

    if isinstance(value, str):
        value = value.replace("\n", " ")
        value = re.sub(r"\s+", " ", value).strip()

        if value == "":
            return None

    return value


def normalize_label(value):
    value = clean_cell(value)

    if value is None:
        return ""

    text = str(value).upper().strip()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()



## 6. Detect block start

Daripada hardcode A:G dan I:O, kita mencari `NAMA KOMPONEN`.

Karena `NAMA KOMPONEN` ada di kolom kedua setiap block, maka:

```python
block_start = nama_komponen_col - 1
```


In [ ]:

def detect_block_start_columns(df):
    starts = []

    for r in range(df.shape[0]):
        for c in range(df.shape[1]):
            if normalize_label(df.iat[r, c]) == "NAMA KOMPONEN":
                start_col = c - 1
                if start_col >= 0:
                    starts.append(start_col)

    return sorted(set(starts))


if EXCEL_FILE.exists():
    block_starts = detect_block_start_columns(raw)
    print("Detected block starts:", block_starts)



## 7. Preview setiap block


In [ ]:

if EXCEL_FILE.exists():
    for i, start_col in enumerate(block_starts, 1):
        print("=" * 80)
        print(f"BLOCK {i} | columns {start_col}..{start_col + BLOCK_WIDTH - 1}")

        display(
            raw.iloc[
                :50,
                start_col:start_col + BLOCK_WIDTH
            ]
        )



# PART B — Parse metadata lokomotif

Metadata dicari berdasarkan label, bukan berdasarkan nomor row absolut.


In [ ]:

METADATA_LABELS = {
    "no_seri_lokomotif": ["NO SERI LOKOMOTIF"],
    "dipo_induk": ["DIPO INDUK"],
    "jenis_perawatan": ["JENIS PERAWATAN"],
    "program_bulan": ["PROGRAM BULAN"],
    "masuk": ["MASUK"],
    "keluar": ["KELUAR"],
}


def label_matches(value, variants):
    normalized = normalize_label(value)
    normalized_variants = {
        normalize_label(v)
        for v in variants
    }
    return normalized in normalized_variants


def first_value_to_right(
    df,
    row_idx,
    label_col,
    block_end
):
    for c in range(label_col + 1, block_end + 1):
        value = clean_cell(df.iat[row_idx, c])

        if value is None:
            continue

        if str(value).strip() == ":":
            continue

        return value

    return None


def parse_metadata(
    df,
    block_start,
    block_width=7,
    search_rows=20
):
    block_end = min(
        block_start + block_width - 1,
        df.shape[1] - 1
    )

    result = {
        key: None
        for key in METADATA_LABELS
    }

    max_row = min(search_rows, df.shape[0])

    for r in range(max_row):
        for c in range(block_start, block_end + 1):
            cell = df.iat[r, c]

            for key, variants in METADATA_LABELS.items():
                if result[key] is not None:
                    continue

                if label_matches(cell, variants):
                    result[key] = first_value_to_right(
                        df,
                        r,
                        c,
                        block_end
                    )

    return result



## 8. Test metadata parsing


In [ ]:

if EXCEL_FILE.exists():
    for i, start_col in enumerate(block_starts, 1):
        metadata = parse_metadata(
            raw,
            start_col,
            BLOCK_WIDTH
        )

        print("=" * 70)
        print("BLOCK", i)

        for key, value in metadata.items():
            print(f"{key:22} = {value}")



## 9. Parse tanggal

Source menggunakan format Indonesia `DD/MM/YYYY`, maka `dayfirst=True`.


In [ ]:

def parse_date(value):
    value = clean_cell(value)

    if value is None:
        return None

    parsed = pd.to_datetime(
        value,
        errors="coerce",
        dayfirst=True
    )

    if pd.isna(parsed):
        return None

    return parsed.date()



# PART C — Parse detail komponen

Struktur 7 kolom:

```text
0 NO.
1 NAMA KOMPONEN
2 ASAL - KODE CETAK
3 ASAL - NO.MANUF
4 PENGGANTI - KODE CETAK
5 PENGGANTI - NO.MANUF
6 KET.
```


In [ ]:

COMPONENT_COLUMNS = [
    "component_no",
    "component_name",
    "asal_kode_cetak",
    "asal_no_manuf",
    "pengganti_kode_cetak",
    "pengganti_no_manuf",
    "keterangan"
]



## 10. Find header row


In [ ]:

def find_component_header_row(
    df,
    block_start,
    block_width=7
):
    block_end = min(
        block_start + block_width,
        df.shape[1]
    )

    for r in range(df.shape[0]):
        for c in range(block_start, block_end):
            if normalize_label(df.iat[r, c]) == "NAMA KOMPONEN":
                return r

    return None



## 11. Forward-fill merged rows

Contoh `INJECTION NOZZLE` menggunakan merged cells secara vertikal.

Pandas biasanya membaca:

```text
8    INJECTION NOZZLE
NaN  NaN
NaN  NaN
```

Maka `component_no` dan `component_name` perlu `ffill()`.


In [ ]:

def row_is_empty(values):
    return all(
        clean_cell(v) is None
        for v in values
    )


def parse_components(
    df,
    block_start,
    block_width=7
):
    header_row = find_component_header_row(
        df,
        block_start,
        block_width
    )

    if header_row is None:
        return pd.DataFrame(
            columns=COMPONENT_COLUMNS
        )

    # Dua tingkat header:
    # row utama + row subheader
    data_start = header_row + 2

    rows = []

    for r in range(data_start, df.shape[0]):
        values = []

        for c in range(
            block_start,
            block_start + block_width
        ):
            if c < df.shape[1]:
                values.append(clean_cell(df.iat[r, c]))
            else:
                values.append(None)

        if row_is_empty(values):
            continue

        normalized = [
            normalize_label(v)
            for v in values
        ]

        if "NAMA KOMPONEN" in normalized:
            continue

        rows.append(values)

    result = pd.DataFrame(
        rows,
        columns=COMPONENT_COLUMNS
    )

    if result.empty:
        return result

    result["component_no"] = (
        result["component_no"]
        .replace("", np.nan)
        .ffill()
    )

    result["component_name"] = (
        result["component_name"]
        .replace("", np.nan)
        .ffill()
    )

    # Minimal satu dari field equipment / keterangan harus terisi
    detail_cols = [
        "asal_kode_cetak",
        "asal_no_manuf",
        "pengganti_kode_cetak",
        "pengganti_no_manuf",
        "keterangan"
    ]

    result = result[
        result[detail_cols]
        .notna()
        .any(axis=1)
    ].reset_index(drop=True)

    return result



## 12. Test component parsing


In [ ]:

if EXCEL_FILE.exists():
    for i, start_col in enumerate(block_starts, 1):
        components = parse_components(
            raw,
            start_col,
            BLOCK_WIDTH
        )

        print("=" * 80)
        print(
            f"BLOCK {i} | rows={len(components)}"
        )

        display(components.head(50))



# PART D — Parse satu block menjadi parent + children


In [ ]:

def parse_locomotive_block(
    df,
    source_sheet,
    block_index,
    block_start,
    block_width=7
):
    metadata = parse_metadata(
        df,
        block_start,
        block_width
    )

    metadata["masuk"] = parse_date(
        metadata["masuk"]
    )

    metadata["keluar"] = parse_date(
        metadata["keluar"]
    )

    event = {
        "source_sheet": source_sheet,
        "block_index": block_index,
        **metadata
    }

    components = parse_components(
        df,
        block_start,
        block_width
    )

    if not components.empty:
        components.insert(
            0,
            "block_index",
            block_index
        )

        components.insert(
            0,
            "source_sheet",
            source_sheet
        )

    return event, components



## 13. Parse satu worksheet


In [ ]:

def parse_sheet(
    excel_path,
    sheet_name,
    block_width=7
):
    df = pd.read_excel(
        excel_path,
        sheet_name=sheet_name,
        header=None,
        dtype=object,
        engine="openpyxl"
    )

    starts = detect_block_start_columns(df)

    events = []
    component_frames = []

    for block_index, start_col in enumerate(
        starts,
        start=1
    ):
        event, components = parse_locomotive_block(
            df=df,
            source_sheet=sheet_name,
            block_index=block_index,
            block_start=start_col,
            block_width=block_width
        )

        events.append(event)

        if not components.empty:
            component_frames.append(components)

    events_df = pd.DataFrame(events)

    components_df = (
        pd.concat(
            component_frames,
            ignore_index=True
        )
        if component_frames
        else pd.DataFrame(
            columns=[
                "source_sheet",
                "block_index",
                *COMPONENT_COLUMNS
            ]
        )
    )

    return events_df, components_df



## 14. Parse seluruh workbook


In [ ]:

def parse_workbook(
    excel_path,
    sheet_names=None,
    block_width=7
):
    xls = pd.ExcelFile(
        excel_path,
        engine="openpyxl"
    )

    if sheet_names is None:
        sheet_names = xls.sheet_names

    all_events = []
    all_components = []

    for sheet_name in sheet_names:
        print("Processing:", sheet_name)

        events, components = parse_sheet(
            excel_path,
            sheet_name,
            block_width
        )

        if not events.empty:
            all_events.append(events)

        if not components.empty:
            all_components.append(components)

    events_df = (
        pd.concat(
            all_events,
            ignore_index=True
        )
        if all_events
        else pd.DataFrame()
    )

    components_df = (
        pd.concat(
            all_components,
            ignore_index=True
        )
        if all_components
        else pd.DataFrame()
    )

    return events_df, components_df


if EXCEL_FILE.exists():
    events_df, components_df = parse_workbook(
        EXCEL_FILE,
        selected_sheets,
        BLOCK_WIDTH
    )

    print("Maintenance events:", len(events_df))
    print("Component rows:", len(components_df))



## 15. Inspect normalized maintenance events


In [ ]:

if EXCEL_FILE.exists():
    display(events_df)



## 16. Inspect normalized components


In [ ]:

if EXCEL_FILE.exists():
    display(components_df.head(200))



# PART E — Validasi

Event minimal harus mempunyai `no_seri_lokomotif`.

Setiap component row setelah forward-fill seharusnya mempunyai `component_name`.


In [ ]:

if EXCEL_FILE.exists():
    invalid_events = events_df[
        events_df["no_seri_lokomotif"].isna()
    ]

    print(
        "Invalid events:",
        len(invalid_events)
    )

    if len(invalid_events):
        display(invalid_events)

    invalid_components = components_df[
        components_df["component_name"].isna()
    ]

    print(
        "Components without name:",
        len(invalid_components)
    )

    if len(invalid_components):
        display(invalid_components)



## 17. Exact duplicate check

Nama seperti `INJECTION NOZZLE` boleh muncul berkali-kali.

Yang dicek di sini adalah duplicate penuh untuk row yang sama.


In [ ]:

if EXCEL_FILE.exists() and not components_df.empty:
    duplicate_cols = [
        "source_sheet",
        "block_index",
        "component_no",
        "component_name",
        "asal_kode_cetak",
        "asal_no_manuf",
        "pengganti_kode_cetak",
        "pengganti_no_manuf",
        "keterangan"
    ]

    duplicates = components_df[
        components_df.duplicated(
            subset=duplicate_cols,
            keep=False
        )
    ]

    print("Exact duplicate rows:", len(duplicates))

    if len(duplicates):
        display(duplicates)



# PART F — SQLite schema

Kita simpan dua tabel:

```text
maintenance_events
equipment_components
```

Relasi:

```text
maintenance_events.id
    -> equipment_components.maintenance_event_id
```


In [ ]:

def get_connection():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.row_factory = sqlite3.Row
    return conn



## 18. Create database tables


In [ ]:

CREATE_EVENTS_SQL = '''
CREATE TABLE IF NOT EXISTS maintenance_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    source_file TEXT NOT NULL,
    source_sheet TEXT NOT NULL,
    block_index INTEGER NOT NULL,

    no_seri_lokomotif TEXT,
    dipo_induk TEXT,
    jenis_perawatan TEXT,
    program_bulan TEXT,

    masuk TEXT,
    keluar TEXT,

    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,

    UNIQUE (
        source_file,
        source_sheet,
        block_index
    )
);
'''

CREATE_COMPONENTS_SQL = '''
CREATE TABLE IF NOT EXISTS equipment_components (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    maintenance_event_id INTEGER NOT NULL,

    component_no TEXT,
    component_name TEXT,

    asal_kode_cetak TEXT,
    asal_no_manuf TEXT,

    pengganti_kode_cetak TEXT,
    pengganti_no_manuf TEXT,

    keterangan TEXT,

    FOREIGN KEY (
        maintenance_event_id
    )
    REFERENCES maintenance_events(id)
    ON DELETE CASCADE
);
'''

CREATE_LOG_SQL = '''
CREATE TABLE IF NOT EXISTS ingestion_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    source_file TEXT NOT NULL,
    started_at TEXT NOT NULL,
    finished_at TEXT,

    event_count INTEGER DEFAULT 0,
    component_count INTEGER DEFAULT 0,

    status TEXT NOT NULL,
    error_message TEXT
);
'''

with get_connection() as conn:
    conn.execute(CREATE_EVENTS_SQL)
    conn.execute(CREATE_COMPONENTS_SQL)
    conn.execute(CREATE_LOG_SQL)

print("SQLite schema ready.")



# PART G — Transactional ingestion

Strategi:

```text
UPSERT maintenance event
DELETE old child components
INSERT current child components
COMMIT
```

Jika error terjadi, transaction di-rollback.


In [ ]:

def sqlite_value(value):
    if value is None:
        return None

    if pd.isna(value):
        return None

    if hasattr(value, "isoformat"):
        try:
            return value.isoformat()
        except Exception:
            pass

    return str(value)


In [ ]:

UPSERT_EVENT_SQL = '''
INSERT INTO maintenance_events (
    source_file,
    source_sheet,
    block_index,
    no_seri_lokomotif,
    dipo_induk,
    jenis_perawatan,
    program_bulan,
    masuk,
    keluar,
    created_at,
    updated_at
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)

ON CONFLICT (
    source_file,
    source_sheet,
    block_index
)
DO UPDATE SET
    no_seri_lokomotif = excluded.no_seri_lokomotif,
    dipo_induk = excluded.dipo_induk,
    jenis_perawatan = excluded.jenis_perawatan,
    program_bulan = excluded.program_bulan,
    masuk = excluded.masuk,
    keluar = excluded.keluar,
    updated_at = excluded.updated_at;
'''


INSERT_COMPONENT_SQL = '''
INSERT INTO equipment_components (
    maintenance_event_id,
    component_no,
    component_name,
    asal_kode_cetak,
    asal_no_manuf,
    pengganti_kode_cetak,
    pengganti_no_manuf,
    keterangan
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?);
'''



## 19. Parent upsert helper


In [ ]:

def upsert_event(
    conn,
    source_file,
    event_row
):
    now = datetime.now(
        timezone.utc
    ).isoformat()

    values = (
        source_file,
        sqlite_value(event_row["source_sheet"]),
        int(event_row["block_index"]),
        sqlite_value(event_row["no_seri_lokomotif"]),
        sqlite_value(event_row["dipo_induk"]),
        sqlite_value(event_row["jenis_perawatan"]),
        sqlite_value(event_row["program_bulan"]),
        sqlite_value(event_row["masuk"]),
        sqlite_value(event_row["keluar"]),
        now,
        now
    )

    conn.execute(
        UPSERT_EVENT_SQL,
        values
    )

    event_id = conn.execute(
        """
        SELECT id
        FROM maintenance_events
        WHERE source_file = ?
          AND source_sheet = ?
          AND block_index = ?
        """,
        (
            source_file,
            event_row["source_sheet"],
            int(event_row["block_index"])
        )
    ).fetchone()["id"]

    return event_id



## 20. Replace child components


In [ ]:

def replace_components(
    conn,
    event_id,
    component_rows
):
    conn.execute(
        """
        DELETE FROM equipment_components
        WHERE maintenance_event_id = ?
        """,
        (event_id,)
    )

    records = []

    for _, row in component_rows.iterrows():
        records.append(
            (
                event_id,
                sqlite_value(row["component_no"]),
                sqlite_value(row["component_name"]),
                sqlite_value(row["asal_kode_cetak"]),
                sqlite_value(row["asal_no_manuf"]),
                sqlite_value(row["pengganti_kode_cetak"]),
                sqlite_value(row["pengganti_no_manuf"]),
                sqlite_value(row["keterangan"])
            )
        )

    if records:
        conn.executemany(
            INSERT_COMPONENT_SQL,
            records
        )

    return len(records)



## 21. Full ingestion function


In [ ]:

def ingest_to_sqlite(
    events_df,
    components_df,
    source_file
):
    started_at = datetime.now(
        timezone.utc
    ).isoformat()

    with get_connection() as conn:
        cursor = conn.execute(
            """
            INSERT INTO ingestion_history (
                source_file,
                started_at,
                status
            )
            VALUES (?, ?, ?)
            """,
            (
                source_file,
                started_at,
                "RUNNING"
            )
        )

        log_id = cursor.lastrowid
        conn.commit()

        total_components = 0

        try:
            conn.execute("BEGIN")

            for _, event in events_df.iterrows():
                event_id = upsert_event(
                    conn,
                    source_file,
                    event
                )

                matching = components_df[
                    (
                        components_df["source_sheet"]
                        == event["source_sheet"]
                    )
                    &
                    (
                        components_df["block_index"]
                        == event["block_index"]
                    )
                ]

                total_components += replace_components(
                    conn,
                    event_id,
                    matching
                )

            conn.commit()

            finished_at = datetime.now(
                timezone.utc
            ).isoformat()

            conn.execute(
                """
                UPDATE ingestion_history
                SET
                    finished_at = ?,
                    event_count = ?,
                    component_count = ?,
                    status = ?
                WHERE id = ?
                """,
                (
                    finished_at,
                    len(events_df),
                    total_components,
                    "SUCCESS",
                    log_id
                )
            )

            conn.commit()

            return {
                "status": "SUCCESS",
                "events": len(events_df),
                "components": total_components,
                "log_id": log_id
            }

        except Exception as exc:
            conn.rollback()

            finished_at = datetime.now(
                timezone.utc
            ).isoformat()

            conn.execute(
                """
                UPDATE ingestion_history
                SET
                    finished_at = ?,
                    status = ?,
                    error_message = ?
                WHERE id = ?
                """,
                (
                    finished_at,
                    "FAILED",
                    str(exc),
                    log_id
                )
            )

            conn.commit()

            raise



## 22. Jalankan ingestion


In [ ]:

if EXCEL_FILE.exists():
    result = ingest_to_sqlite(
        events_df,
        components_df,
        EXCEL_FILE.name
    )

    print(result)



# PART H — Query validation


In [ ]:

with get_connection() as conn:
    db_events = pd.read_sql_query(
        """
        SELECT *
        FROM maintenance_events
        ORDER BY id;
        """,
        conn
    )

display(db_events)


In [ ]:

with get_connection() as conn:
    db_components = pd.read_sql_query(
        """
        SELECT *
        FROM equipment_components
        ORDER BY maintenance_event_id, id;
        """,
        conn
    )

display(db_components.head(200))



## 23. JOIN parent + child

Hasil ini cocok untuk reporting / export.


In [ ]:

JOIN_QUERY = """
SELECT
    e.id AS maintenance_event_id,
    e.no_seri_lokomotif,
    e.dipo_induk,
    e.jenis_perawatan,
    e.program_bulan,
    e.masuk,
    e.keluar,

    c.component_no,
    c.component_name,
    c.asal_kode_cetak,
    c.asal_no_manuf,
    c.pengganti_kode_cetak,
    c.pengganti_no_manuf,
    c.keterangan

FROM maintenance_events e

LEFT JOIN equipment_components c
    ON c.maintenance_event_id = e.id

ORDER BY
    e.id,
    c.id;
"""

with get_connection() as conn:
    report_df = pd.read_sql_query(
        JOIN_QUERY,
        conn
    )

display(report_df.head(200))



## 24. Search satu lokomotif


In [ ]:

LOCOMOTIVE = "CC 201 77 11"

with get_connection() as conn:
    locomotive_df = pd.read_sql_query(
        """
        SELECT
            e.no_seri_lokomotif,
            e.dipo_induk,
            e.jenis_perawatan,
            e.masuk,
            e.keluar,

            c.component_no,
            c.component_name,
            c.asal_kode_cetak,
            c.asal_no_manuf,
            c.pengganti_kode_cetak,
            c.pengganti_no_manuf,
            c.keterangan

        FROM maintenance_events e

        JOIN equipment_components c
            ON c.maintenance_event_id = e.id

        WHERE e.no_seri_lokomotif = ?

        ORDER BY
            e.masuk,
            c.id;
        """,
        conn,
        params=(LOCOMOTIVE,)
    )

display(locomotive_df)



## 25. Ingestion history


In [ ]:

with get_connection() as conn:
    history_df = pd.read_sql_query(
        """
        SELECT *
        FROM ingestion_history
        ORDER BY id DESC;
        """,
        conn
    )

display(history_df)



# PART I — Export normalized result untuk debugging

Sebelum memakai database untuk production, sangat disarankan mengecek hasil parser dalam bentuk Excel datar.


In [ ]:

if EXCEL_FILE.exists():
    DEBUG_OUTPUT = Path(
        "parsed_lokomotif_debug.xlsx"
    )

    with pd.ExcelWriter(
        DEBUG_OUTPUT,
        engine="openpyxl"
    ) as writer:

        events_df.to_excel(
            writer,
            sheet_name="maintenance_events",
            index=False
        )

        components_df.to_excel(
            writer,
            sheet_name="equipment_components",
            index=False
        )

    print(
        "Debug file:",
        DEBUG_OUTPUT.resolve()
    )



# Expected result

Untuk screenshot Anda, output `events_df` secara konsep akan seperti:

```text
source_sheet | block_index | no_seri_lokomotif | dipo_induk | jenis_perawatan | masuk      | keluar
Sheet1       | 1           | CC 201 77 11      | SDT        | P48              | 2025-04-10 | 2025-05-01
Sheet1       | 2           | CC 201 92 12      | CPN        | P48              | 2025-04-11 | 2025-05-01
```

Sedangkan `components_df` untuk merged section seperti `INJECTION NOZZLE` akan menjadi:

```text
component_no | component_name    | asal_kode_cetak | asal_no_manuf | ...
8            | INJECTION NOZZLE  | IN-1399749      | 0101031337    | ...
8            | INJECTION NOZZLE  | IN-1155658      | 0101030X51    | ...
8            | INJECTION NOZZLE  | IN-1149840      | 0101030667    | ...
```

Jadi walaupun Excel memakai merged cells, hasil database tetap satu row per equipment.



# Catatan penting

Notebook ini sengaja **tidak hardcode row number** seperti row 18, row 19, dan seterusnya.

Parser menggunakan marker:

```text
NAMA KOMPONEN
NO.SERI LOKOMOTIF
DIPO INDUK
JENIS PERAWATAN
PROGRAM BULAN
MASUK
KELUAR
```

Sehingga lebih tahan terhadap:

- hidden rows
- blank rows tambahan
- perubahan posisi vertikal
- multiple blocks dalam satu sheet

Yang masih diasumsikan tetap adalah urutan 7 kolom detail komponen.
